#Gold Table Statistics

In [0]:


%sql
Create or replace table YAGoldModel.Gold_table_stats_temp as (
 --Generated SQL Start Tag
 SELECT 'DimCustomer' AS TableName,
   COUNT(*) RecordCount,
   CAST(Audit_CreatedDateTime AS DATE) AS Audit_CreatedDateTime,
   (SELECT COUNT(*) FROM IDENTIFIER(var_destination_catalog_name||'.YAGoldModel'||'.DimCustomer')) AS AllTimeRecordCount,
   (SELECT COUNT(*) FROM IDENTIFIER(var_destination_catalog_name||'.YAGoldModel'||'.DimCustomer')
   WHERE Audit_CreatedDateTime >=Date_add( current_timestamp(), -30)) Last30DaysRecordCount
 FROM IDENTIFIER(var_destination_catalog_name||'.YAGoldModel'||'.DimCustomer') GROUP BY CAST(Audit_CreatedDateTime AS DATE)
 UNION ALL
 SELECT 'DimProduct' AS TableName,
   COUNT(*) RecordCount,
   CAST(Audit_CreatedDateTime AS DATE) AS Audit_CreatedDateTime,
   (SELECT COUNT(*) FROM IDENTIFIER(var_destination_catalog_name||'.YAGoldModel'||'.DimProduct')) AS AllTimeRecordCount,
   (SELECT COUNT(*) FROM IDENTIFIER(var_destination_catalog_name||'.YAGoldModel'||'.DimProduct')
   WHERE Audit_CreatedDateTime >=Date_add( current_timestamp(), -30)) Last30DaysRecordCount
 FROM IDENTIFIER(var_destination_catalog_name||'.YAGoldModel'||'.DimProduct') GROUP BY CAST(Audit_CreatedDateTime AS DATE)
----Generated SQL End Tag
)

num_affected_rows,num_inserted_rows


In [0]:

%sql
CREATE OR REPLACE table YAGoldModel.Gold_table_stats AS (
  WITH DistinctTables AS (
    SELECT 
      TableName, 
      AllTimeRecordCount, 
      Last30DaysRecordCount,
      DENSE_RANK() OVER (ORDER BY AllTimeRecordCount DESC, TableName) AllTimeRank,
      DENSE_RANK() OVER (ORDER BY Last30DaysRecordCount DESC, TableName) Last30DaysRank
    FROM YAGoldModel.Gold_table_stats_temp
    GROUP BY TableName, AllTimeRecordCount, Last30DaysRecordCount 
  )
  
  ,DistinctCreateDates AS
  (
    SELECT DISTINCT Audit_CreatedDateTime FROM YAGoldModel.Gold_table_stats_temp
    UNION ALL
    SELECT IFNULL(DATE_ADD(MIN(Audit_CreatedDateTime),-1),CAST('2001-01-01' AS DATE)) FROM YAGoldModel.Gold_table_stats_temp
  )
  
  , TableAndAllCreateDates AS (
    SELECT 
      t1.TableName, 
      t1.AllTimeRecordCount, 
      t1.AllTimeRank, 
      t1.Last30DaysRecordCount, 
      t1.Last30DaysRank, 
      t2.Audit_CreatedDateTime 
    FROM DistinctTables t1
    CROSS JOIN DistinctCreateDates t2
    ON 1=1
  )
  , ToDateTotals AS (
    SELECT 
      t3.TableName
      , t3.AllTimeRecordCount
      , t3.AllTimeRank
      , t3.Last30DaysRecordCount
      , t3.Last30DaysRank
      , t3.Audit_CreatedDateTime
      , IFNULL(sum(t4.RecordCount),0) ToDateRecordCount
    FROM TableAndAllCreateDates t3
    LEFT OUTER JOIN (
      SELECT TableName, RecordCount, Audit_CreatedDateTime FROM YAGoldModel.Gold_table_stats_temp
    ) t4
    ON t3.TableName = t4.TableName
    AND t3.Audit_CreatedDateTime >=t4.Audit_CreatedDateTime
    GROUP BY t3.TableName, t3.AllTimeRecordCount, t3.Last30DaysRecordCount, t3.Last30DaysRank, t3.Audit_CreatedDateTime, t3.AllTimeRank
  )
 
  SELECT 
    t6.TableName, 
    t6.Audit_CreatedDateTime as LoadDate, 
    IFNULL(t7.RecordCount,0) AS RecordCount, 
    t6.ToDateRecordCount, 
    t6.AllTimeRecordCount,
    t6.Last30DaysRecordCount,
    t6.Last30DaysRank,
    t6.AllTimeRank
  
 
  FROM ToDateTotals t6
  LEFT OUTER JOIN YAGoldModel.Gold_table_stats_temp t7
  ON t6.TableName = t7.TableName
  AND t6.Audit_CreatedDateTime = t7.Audit_CreatedDateTime

)



num_affected_rows,num_inserted_rows


In [0]:
%sql
drop table YAGoldModel.Gold_table_stats_temp

In [0]:
%sql
select count(*) from YAGoldModel.Gold_table_stats

count(1)
4


##Records Total
####(all-time top 20 tables)

In [0]:
display(spark.sql("select DISTINCT TableName, AllTimeRecordCount AS RecordCount, AllTimeRank  from YAGoldModel.Gold_table_stats WHERE AllTimeRank <= 20 ORDER BY AllTimeRank"))

TableName,RecordCount,AllTimeRank
BusinessEntity,20777,1
AddressType,6,2


##Records Loaded Last 30 days
####(top 20 tables in the last 30 days)

In [0]:
display(spark.sql("select DISTINCT TableName, Last30DaysRecordCount AS RecordCount, Last30DaysRank  from YAGoldModel.Gold_table_stats WHERE Last30DaysRank <= 20 ORDER BY Last30DaysRank"))

TableName,RecordCount,Last30DaysRank
BusinessEntity,20777,1
AddressType,6,2


##Record Growth
####(top 20 tables in the last 30 days)

In [0]:
display(spark.sql("select ToDateRecordCount AS Records, TableName, LoadDate from YAGoldModel.Gold_table_stats where LoadDate >=Date_add( current_timestamp(), -30) AND Last30DaysRank <= 20 ORDER BY LoadDate, TableName"  ))

Records,TableName,LoadDate
0,AddressType,2022-08-11
0,BusinessEntity,2022-08-11
6,AddressType,2022-08-12
20777,BusinessEntity,2022-08-12


##Records by Load Date
####(top 20 Tables in the Last 30 Days)

In [0]:
display(spark.sql("select TableName, RecordCount, LoadDate as LoadDate from YAGoldModel.Gold_table_stats WHERE LoadDate >=Date_add( current_timestamp(), -30) AND Last30DaysRank <= 20 ORDER BY LoadDate, TableName"))


TableName,RecordCount,LoadDate
AddressType,0,2022-08-11
BusinessEntity,0,2022-08-11
AddressType,6,2022-08-12
BusinessEntity,20777,2022-08-12
